# Matching INE--BANAVIM con criterios conservadores

Este notebook parte de las bases canónicas ya generadas en el notebook principal:

- `ine_fusion_2020_2022.csv`
- `banavim_fusion_2020_2022.csv.gz`

El objetivo no es repetir toda la limpieza, sino auditar los campos disponibles para record linkage y construir una estrategia de matching conservadora.

Dado que BANAVIM no contiene nombre del agresor ni un identificador común con INE, el matching no se interpreta como identificación individual de personas. Se interpreta como una vinculación aproximada entre registros o perfiles institucionales compatibles.

Antes de generar pares candidatos, se revisa especialmente la variable `vinculo_grupo`, porque el campo original `vinculo_victima` no tiene el mismo significado semántico en ambas fuentes.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path("../Data/fusion_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ine_fusion = pd.read_csv(
    OUTPUT_DIR / "ine_fusion_2020_2022.csv",
    encoding="utf-8-sig"
)

bv_fusion = pd.read_csv(
    OUTPUT_DIR / "banavim_fusion_2020_2022.csv.gz",
    encoding="utf-8-sig",
    compression="gzip"
)

print("ine_fusion:", ine_fusion.shape)
print("bv_fusion:", bv_fusion.shape)

display(ine_fusion.head())
display(bv_fusion.head())

ine_fusion: (162, 10)
bv_fusion: (806092, 10)


,id_ine_caso,Nombre,Número De Expediente,fecha_resolucion,anio_resolucion,sexo,estado,municipio,vinculo_victima,vinculo_grupo
0,INE_00000,AARÓN VEGA REA,SRE-PSC-41/2022,2022-04-07,2022,H,GUANAJUATO,NaN,NINGUNA,NINGUNA
1,INE_00001,ABEL TOVILLA CARPIO,IEPC/PE/GJMA/081/2021,2021-11-10,2021,H,CHIAPAS,TEOPISCA,PARES,PARES
2,INE_00002,ADÁN FRAUSTO ARELLANO,TEE-PES-18/2021,2021-07-02,2021,H,NAYARIT,EL NAYAR,JERARQUICA,JERARQUICA_SUBORDINACION
3,INE_00003,ADRIANA AVENDAÑO NIÑO,PES-58/2021,2021-06-04,2021,M,OAXACA,SAN ANDRES ZAUTLA,NINGUNA,NINGUNA
4,INE_00004,ALBERTO ALFONSO MENDOZA CRUZ,JDCI/52/2021 y su acumulado JDCI/56/2021,2021-07-09,2021,H,OAXACA,SAN LORENZO CACAOTEPEC,JERARQUICA,JERARQUICA_SUBORDINACION


,id_bv_caso,Identificador Único,año_base,fecha_registro,anio_registro,sexo,estado,municipio,vinculo_victima,vinculo_grupo
0,BV_0000000,0128900022-2,2020,2020-09-03 08:59:29,2020,H,AGUASCALIENTES,AGUASCALIENTES,EX PAREJA,PAREJA_EXPAREJA
1,BV_0000001,0128900106-2,2020,2020-10-20 09:25:05,2020,H,AGUASCALIENTES,AGUASCALIENTES,CONYUGE O PAREJA,PAREJA_EXPAREJA
2,BV_0000002,0128900132-2,2020,2020-07-02 08:07:24,2020,H,AGUASCALIENTES,AGUASCALIENTES,CONYUGE O PAREJA,PAREJA_EXPAREJA
3,BV_0000003,0128900350-2,2020,2020-01-29 15:48:51,2020,H,AGUASCALIENTES,AGUASCALIENTES,EX PAREJA,PAREJA_EXPAREJA
4,BV_0000004,0128900563-2,2020,2020-12-07 13:50:38,2020,H,AGUASCALIENTES,COSIO,CONYUGE O PAREJA,PAREJA_EXPAREJA


In [4]:
def auditar_origen_grupos(df, nombre_base):
    auditoria = (
        df.groupby(["vinculo_grupo", "vinculo_victima"], dropna=False)
        .size()
        .reset_index(name="frecuencia")
        .sort_values(["vinculo_grupo", "frecuencia"], ascending=[True, False])
    )
    auditoria.insert(0, "base", nombre_base)
    return auditoria

auditoria_vinculos_ine = auditar_origen_grupos(ine_fusion, "INE")
auditoria_vinculos_bv = auditar_origen_grupos(bv_fusion, "BANAVIM")

display(auditoria_vinculos_ine)
display(auditoria_vinculos_bv)

,base,vinculo_grupo,vinculo_victima,frecuencia
0,INE,INSTITUCIONAL_POLITICA,OPOSITOR EN LA CONTIENDA,2
1,INE,JERARQUICA_SUBORDINACION,JERARQUICA,33
2,INE,JERARQUICA_SUBORDINACION,SUBORDINACION,5
3,INE,NINGUNA,NINGUNA,57
5,INE,OTRO,OTRO,3
4,INE,OTRO,OTRA,1
6,INE,PARES,PARES,61


,base,vinculo_grupo,vinculo_victima,frecuencia
2,BANAVIM,COMUNITARIO_SOCIAL,VECINO A,11678
1,BANAVIM,COMUNITARIO_SOCIAL,PROFESOR A,549
0,BANAVIM,COMUNITARIO_SOCIAL,DESCONOCIDO,8
5,BANAVIM,FAMILIAR,HIJO A,23839
6,BANAVIM,FAMILIAR,MADRE O PADRE,23766
4,BANAVIM,FAMILIAR,HERMANO A,15065
8,BANAVIM,FAMILIAR,PADRASTRO O MADRASTRA,5272
12,BANAVIM,FAMILIAR,TIO A,5148
9,BANAVIM,FAMILIAR,PRIMO A,2671
10,BANAVIM,FAMILIAR,SOBRINO A,2468


## Revisión de categorías comparables desde INE hacia BANAVIM

Como el objetivo del matching es enriquecer o vincular registros del INE con BANAVIM, la revisión se centra primero en las categorías que efectivamente aparecen en la base del INE.

La pregunta de esta sección es:

> De las categorías de `vinculo_grupo` que aparecen en INE, ¿qué valores originales de BANAVIM fueron mapeados a esas mismas categorías?

Esto permite evaluar si las categorías compartidas son semánticamente suficientemente compatibles para usarse en matching conservador.

In [5]:
# Categorías de vinculo_grupo presentes en INE y BANAVIM

grupos_ine = set(ine_fusion["vinculo_grupo"].dropna().unique())
grupos_bv = set(bv_fusion["vinculo_grupo"].dropna().unique())

grupos_comunes = sorted(grupos_ine & grupos_bv)
grupos_solo_ine = sorted(grupos_ine - grupos_bv)
grupos_solo_bv = sorted(grupos_bv - grupos_ine)

resumen_grupos = pd.DataFrame({
    "tipo": [
        "grupos en INE",
        "grupos en BANAVIM",
        "grupos comunes",
        "grupos solo INE",
        "grupos solo BANAVIM"
    ],
    "n": [
        len(grupos_ine),
        len(grupos_bv),
        len(grupos_comunes),
        len(grupos_solo_ine),
        len(grupos_solo_bv)
    ],
    "grupos": [
        sorted(grupos_ine),
        sorted(grupos_bv),
        grupos_comunes,
        grupos_solo_ine,
        grupos_solo_bv
    ]
})

display(resumen_grupos)

,tipo,n,grupos
0,grupos en INE,5,"[INSTITUCIONAL_POLITICA, JERARQUICA_SUBORDINAC..."
1,grupos en BANAVIM,8,"[COMUNITARIO_SOCIAL, FAMILIAR, INSTITUCIONAL_P..."
2,grupos comunes,4,"[INSTITUCIONAL_POLITICA, JERARQUICA_SUBORDINAC..."
3,grupos solo INE,1,[NINGUNA]
4,grupos solo BANAVIM,4,"[COMUNITARIO_SOCIAL, FAMILIAR, MISSING, PAREJA..."


In [8]:
# Resumen completo:
# para cada vinculo_grupo creado en INE o BANAVIM,
# mostrar qué valores originales cayeron ahí en cada fuente.

def resumen_valores_por_grupo_todos(df, nombre_base):
    """
    Resume, por cada vinculo_grupo, los valores originales de vinculo_victima
    que cayeron en ese grupo.

    Importante:
    - No elimina missing originales.
    - Convierte NaN en una etiqueta explícita para no confundir:
      * grupo ausente en una fuente
      * valor original faltante dentro de un grupo existente
    """
    tmp = df.copy()

    tmp["vinculo_victima_mostrar"] = (
        tmp["vinculo_victima"]
        .astype("object")
        .where(tmp["vinculo_victima"].notna(), "<<MISSING_ORIGINAL>>")
    )

    resumen = (
        tmp.groupby("vinculo_grupo", dropna=False)
        .agg(
            registros=("vinculo_victima_mostrar", "size"),
            n_valores_originales=("vinculo_victima_mostrar", "nunique"),
            valores_originales=(
                "vinculo_victima_mostrar",
                lambda s: sorted(s.unique())
            )
        )
        .reset_index()
    )

    resumen = resumen.rename(columns={
        "registros": f"registros_{nombre_base}",
        "n_valores_originales": f"n_valores_originales_{nombre_base}",
        "valores_originales": f"valores_originales_{nombre_base}"
    })

    return resumen


resumen_grupos_ine = resumen_valores_por_grupo_todos(
    ine_fusion,
    "ine"
)

resumen_grupos_bv = resumen_valores_por_grupo_todos(
    bv_fusion,
    "banavim"
)

comparacion_todos_los_grupos = resumen_grupos_ine.merge(
    resumen_grupos_bv,
    on="vinculo_grupo",
    how="outer"
)

# Indicar en qué base aparece cada grupo
comparacion_todos_los_grupos["aparece_en_ine"] = (
    comparacion_todos_los_grupos["registros_ine"].notna()
)

comparacion_todos_los_grupos["aparece_en_banavim"] = (
    comparacion_todos_los_grupos["registros_banavim"].notna()
)

# Rellenar conteos numéricos, pero NO rellenar listas,
# porque NaN en listas significa "ese grupo no existe en esa fuente"
cols_num = [
    "registros_ine",
    "n_valores_originales_ine",
    "registros_banavim",
    "n_valores_originales_banavim"
]

comparacion_todos_los_grupos[cols_num] = (
    comparacion_todos_los_grupos[cols_num]
    .fillna(0)
    .astype(int)
)

# Reordenar columnas
comparacion_todos_los_grupos = comparacion_todos_los_grupos[
    [
        "vinculo_grupo",
        "aparece_en_ine",
        "aparece_en_banavim",
        "registros_ine",
        "n_valores_originales_ine",
        "valores_originales_ine",
        "registros_banavim",
        "n_valores_originales_banavim",
        "valores_originales_banavim"
    ]
].sort_values("vinculo_grupo")

display(comparacion_todos_los_grupos)

,vinculo_grupo,aparece_en_ine,aparece_en_banavim,registros_ine,n_valores_originales_ine,valores_originales_ine,registros_banavim,n_valores_originales_banavim,valores_originales_banavim
0,COMUNITARIO_SOCIAL,False,True,0,0,NaN,12235,3,"[DESCONOCIDO, PROFESOR A, VECINO A]"
1,FAMILIAR,False,True,0,0,NaN,82646,10,"[ABUELO A, HERMANO A, HIJO A, MADRE O PADRE, N..."
2,INSTITUCIONAL_POLITICA,True,True,2,1,[OPOSITOR EN LA CONTIENDA],515,1,[SERVIDOR PUBLICO]
3,JERARQUICA_SUBORDINACION,True,True,38,2,"[JERARQUICA, SUBORDINACION]",1927,1,[JEFE A O PATRON A]
4,MISSING,False,True,0,0,NaN,209481,2,"[<<MISSING_ORIGINAL>>, SELECCIONE]"
5,NINGUNA,True,False,57,1,[NINGUNA],0,0,NaN
6,OTRO,True,True,4,2,"[OTRA, OTRO]",31139,2,"[OTRA RELACION, OTRO]"
7,PAREJA_EXPAREJA,False,True,0,0,NaN,466234,4,"[CONCUBINA, CONYUGE O PAREJA, EX PAREJA, NOVIO A]"
8,PARES,True,True,61,1,[PARES],1915,1,[COMPANERO A]


In [10]:
# Valores originales que quedaron en REVISAR

def revisar_grupo(df, nombre_base, grupo="REVISAR"):
    tmp = df[df["vinculo_grupo"].eq(grupo)].copy()

    tmp["vinculo_victima_mostrar"] = (
        tmp["vinculo_victima"]
        .astype("object")
        .where(tmp["vinculo_victima"].notna(), "<<MISSING_ORIGINAL>>")
    )

    out = (
        tmp["vinculo_victima_mostrar"]
        .value_counts(dropna=False)
        .reset_index()
    )

    out.columns = ["valor_original", "frecuencia"]
    out.insert(0, "base", nombre_base)
    out.insert(1, "vinculo_grupo", grupo)

    return out


revisar_ine = revisar_grupo(ine_fusion, "INE", "REVISAR")
revisar_bv = revisar_grupo(bv_fusion, "BANAVIM", "REVISAR")

revisar_total = pd.concat(
    [revisar_ine, revisar_bv],
    ignore_index=True
)

display(revisar_total)

,base,vinculo_grupo,valor_original,frecuencia


In [11]:
# Valores originales que quedaron en MISSING

missing_ine = revisar_grupo(ine_fusion, "INE", "MISSING")
missing_bv = revisar_grupo(bv_fusion, "BANAVIM", "MISSING")

missing_total = pd.concat(
    [missing_ine, missing_bv],
    ignore_index=True
)

display(missing_total)

,base,vinculo_grupo,valor_original,frecuencia
0,BANAVIM,MISSING,<<MISSING_ORIGINAL>>,130927
1,BANAVIM,MISSING,SELECCIONE,78554


## Clasificación definitiva de vínculo para matching

Después de auditar los valores originales de `vinculo_victima`, se decidió que la clasificación operativa para matching no dependerá de la variable intermedia `vinculo_grupo`.

La variable `vinculo_grupo` fue útil durante la etapa de exploración y auditoría semántica, pero para el procedimiento final de matching se construye directamente una nueva variable:

- `vinculo_match`
- `confianza_vinculo_match`

Esta decisión simplifica la trazabilidad del notebook: el vínculo usado para matching se deriva directamente del valor original normalizado `vinculo_victima`, con reglas explícitas por fuente.

Se definieron tres niveles:

- `ALTA`: correspondencia semántica fuerte.
- `MEDIA`: correspondencia residual o débil, útil para revisión.
- `NO_APTO`: valores sin contraparte clara o no informativos.

Categorías de alta confianza:

| Fuente | Valor original | `vinculo_match` |
|---|---|---|
| INE | `PARES` | `PARES` |
| BANAVIM | `COMPANERO A` | `PARES` |
| INE | `JERARQUICA`, `SUBORDINACION` | `JERARQUICA_SUBORDINACION` |
| BANAVIM | `JEFE A O PATRON A` | `JERARQUICA_SUBORDINACION` |

Categorías de confianza media:

| Fuente | Valor original | `vinculo_match` |
|---|---|---|
| INE | `OTRO`, `OTRA` | `OTRO_RESIDUAL` |
| BANAVIM | `OTRO`, `OTRA RELACION` | `OTRO_RESIDUAL` |
| INE | `NINGUNA` | `SIN_RELACION_IDENTIFICABLE` |
| BANAVIM | `DESCONOCIDO` | `SIN_RELACION_IDENTIFICABLE` |

Las demás categorías se clasifican como `NO_APTO_MATCH`, porque no tienen una correspondencia suficientemente clara entre fuentes o no aportan información útil para el matching.

En las bases finales de matching no se conservará `vinculo_grupo`; se conservarán únicamente `vinculo_victima`, `vinculo_match` y `confianza_vinculo_match`.

In [18]:
# Variable operativa de vínculo para matching por niveles de confianza
# Versión definitiva: se construye directamente desde vinculo_victima,
# sin depender de vinculo_grupo.

def clasificar_vinculo_match(valor, fuente):
    """
    Clasifica el vínculo original normalizado hacia una categoría operativa
    para matching y un nivel de confianza.

    No usa vinculo_grupo. La clasificación se define directamente desde
    los valores originales normalizados de cada fuente.
    """
    if pd.isna(valor):
        return pd.Series({
            "vinculo_match": "NO_APTO_MATCH",
            "confianza_vinculo_match": "NO_APTO"
        })

    v = str(valor).strip().upper()

    # Placeholders o valores no informativos
    if v in {
        "",
        "NAN",
        "NONE",
        "SELECCIONE",
        "SIN DATO",
        "SIN DATOS",
        "NO ESPECIFICADO",
        "NO ESPECIFICADA",
        "NO SE ESPECIFICA"
    }:
        return pd.Series({
            "vinculo_match": "NO_APTO_MATCH",
            "confianza_vinculo_match": "NO_APTO"
        })

    # INE
    if fuente == "INE":
        if v == "PARES":
            return pd.Series({
                "vinculo_match": "PARES",
                "confianza_vinculo_match": "ALTA"
            })

        if v in {"JERARQUICA", "SUBORDINACION"}:
            return pd.Series({
                "vinculo_match": "JERARQUICA_SUBORDINACION",
                "confianza_vinculo_match": "ALTA"
            })

        if v in {"OTRO", "OTRA", "OTRA RELACION"}:
            return pd.Series({
                "vinculo_match": "OTRO_RESIDUAL",
                "confianza_vinculo_match": "MEDIA"
            })

        if v == "NINGUNA":
            return pd.Series({
                "vinculo_match": "SIN_RELACION_IDENTIFICABLE",
                "confianza_vinculo_match": "MEDIA"
            })

    # BANAVIM
    if fuente == "BANAVIM":
        if v in {"COMPANERO A", "COMPANERO", "COMPANERA"}:
            return pd.Series({
                "vinculo_match": "PARES",
                "confianza_vinculo_match": "ALTA"
            })

        if v in {"JEFE A O PATRON A", "JEFE A", "PATRON A", "JEFE", "PATRON"}:
            return pd.Series({
                "vinculo_match": "JERARQUICA_SUBORDINACION",
                "confianza_vinculo_match": "ALTA"
            })

        if v in {"OTRO", "OTRA", "OTRA RELACION", "OTRO TIPO DE RELACION"}:
            return pd.Series({
                "vinculo_match": "OTRO_RESIDUAL",
                "confianza_vinculo_match": "MEDIA"
            })

        if v == "DESCONOCIDO":
            return pd.Series({
                "vinculo_match": "SIN_RELACION_IDENTIFICABLE",
                "confianza_vinculo_match": "MEDIA"
            })

    # Todo lo demás no entra al matching operativo
    return pd.Series({
        "vinculo_match": "NO_APTO_MATCH",
        "confianza_vinculo_match": "NO_APTO"
    })


ine_fusion[["vinculo_match", "confianza_vinculo_match"]] = ine_fusion["vinculo_victima"].apply(
    lambda x: clasificar_vinculo_match(x, "INE")
)

bv_fusion[["vinculo_match", "confianza_vinculo_match"]] = bv_fusion["vinculo_victima"].apply(
    lambda x: clasificar_vinculo_match(x, "BANAVIM")
)

In [19]:
# Auditoría de la nueva clasificación operativa, sin usar vinculo_grupo

def auditoria_vinculo_match(df, nombre_base):
    out = (
        df.groupby(
            ["vinculo_match", "confianza_vinculo_match", "vinculo_victima"],
            dropna=False
        )
        .size()
        .reset_index(name="frecuencia")
        .sort_values(
            ["confianza_vinculo_match", "vinculo_match", "frecuencia"],
            ascending=[True, True, False]
        )
    )
    out.insert(0, "base", nombre_base)
    return out


auditoria_vinculo_match_total = pd.concat(
    [
        auditoria_vinculo_match(ine_fusion, "INE"),
        auditoria_vinculo_match(bv_fusion, "BANAVIM")
    ],
    ignore_index=True
)

display(auditoria_vinculo_match_total)

,base,vinculo_match,confianza_vinculo_match,vinculo_victima,frecuencia
0,INE,JERARQUICA_SUBORDINACION,ALTA,JERARQUICA,33
1,INE,JERARQUICA_SUBORDINACION,ALTA,SUBORDINACION,5
2,INE,PARES,ALTA,PARES,61
3,INE,OTRO_RESIDUAL,MEDIA,OTRO,3
4,INE,OTRO_RESIDUAL,MEDIA,OTRA,1
5,INE,SIN_RELACION_IDENTIFICABLE,MEDIA,NINGUNA,57
6,INE,NO_APTO_MATCH,NO_APTO,OPOSITOR EN LA CONTIENDA,2
7,BANAVIM,JERARQUICA_SUBORDINACION,ALTA,JEFE A O PATRON A,1927
8,BANAVIM,PARES,ALTA,COMPANERO A,1915
9,BANAVIM,OTRO_RESIDUAL,MEDIA,OTRO,31138


## Construcción de bases por nivel de confianza y revisión de municipios

A partir de la clasificación definitiva de `vinculo_victima`, se generan dos escenarios de bases para matching:

- **Alta confianza**: conserva únicamente categorías con equivalencia semántica fuerte.
- **Alta + media confianza**: incorpora además categorías residuales o débiles que podrían ser útiles para revisión individual.

Antes de ejecutar el procedimiento de record linkage, se exportan estas bases para inspección manual y se auditan los municipios disponibles en cada fuente.

La revisión se realiza en dos niveles:

1. `municipio`, para observar coincidencias textuales generales.
2. `estado + municipio`, para evitar confundir municipios homónimos en distintos estados.

Esta segunda revisión es la más importante para decidir si los campos geográficos pueden usarse como parte del matching.

In [20]:
# Crear bases para revisión/matching por nivel de confianza

campos_obligatorios_match = [
    "sexo",
    "estado",
    "municipio",
    "vinculo_match",
    "confianza_vinculo_match"
]

def filtrar_base_match(df, niveles_confianza):
    """
    Filtra registros aptos para matching según:
    - nivel de confianza del vínculo;
    - campos mínimos completos.
    """
    mask = (
        df["confianza_vinculo_match"].isin(niveles_confianza)
        & df["sexo"].notna()
        & df["estado"].notna()
        & df["municipio"].notna()
        & df["vinculo_match"].notna()
    )
    return df.loc[mask].copy()


# Escenario estricto: solo confianza alta
ine_match_alta = filtrar_base_match(
    ine_fusion,
    niveles_confianza=["ALTA"]
)

bv_match_alta = filtrar_base_match(
    bv_fusion,
    niveles_confianza=["ALTA"]
)


# Escenario ampliado: confianza alta + media
ine_match_alta_media = filtrar_base_match(
    ine_fusion,
    niveles_confianza=["ALTA", "MEDIA"]
)

bv_match_alta_media = filtrar_base_match(
    bv_fusion,
    niveles_confianza=["ALTA", "MEDIA"]
)


resumen_bases_match = pd.DataFrame([
    {
        "escenario": "ALTA",
        "base": "INE",
        "registros": len(ine_match_alta),
        "porcentaje_sobre_base_original": round(len(ine_match_alta) / len(ine_fusion) * 100, 2)
    },
    {
        "escenario": "ALTA",
        "base": "BANAVIM",
        "registros": len(bv_match_alta),
        "porcentaje_sobre_base_original": round(len(bv_match_alta) / len(bv_fusion) * 100, 2)
    },
    {
        "escenario": "ALTA_MEDIA",
        "base": "INE",
        "registros": len(ine_match_alta_media),
        "porcentaje_sobre_base_original": round(len(ine_match_alta_media) / len(ine_fusion) * 100, 2)
    },
    {
        "escenario": "ALTA_MEDIA",
        "base": "BANAVIM",
        "registros": len(bv_match_alta_media),
        "porcentaje_sobre_base_original": round(len(bv_match_alta_media) / len(bv_fusion) * 100, 2)
    }
])

display(resumen_bases_match)

,escenario,base,registros,porcentaje_sobre_base_original
0,ALTA,INE,98,60.49
1,ALTA,BANAVIM,3824,0.47
2,ALTA_MEDIA,INE,128,79.01
3,ALTA_MEDIA,BANAVIM,34782,4.31


In [21]:
# Preparar bases finales para exportación
# Se elimina vinculo_grupo porque ya no forma parte de la clasificación definitiva.

columnas_a_quitar_export = [
    "vinculo_grupo"
]

def preparar_export_match(df):
    cols_quitar = [c for c in columnas_a_quitar_export if c in df.columns]
    return df.drop(columns=cols_quitar).copy()


ine_match_alta_export = preparar_export_match(ine_match_alta)
bv_match_alta_export = preparar_export_match(bv_match_alta)

ine_match_alta_media_export = preparar_export_match(ine_match_alta_media)
bv_match_alta_media_export = preparar_export_match(bv_match_alta_media)


# Verificación
for nombre, df in [
    ("ine_match_alta_export", ine_match_alta_export),
    ("bv_match_alta_export", bv_match_alta_export),
    ("ine_match_alta_media_export", ine_match_alta_media_export),
    ("bv_match_alta_media_export", bv_match_alta_media_export)
]:
    print(nombre, df.shape)
    print("¿Tiene vinculo_grupo?", "vinculo_grupo" in df.columns)
    print()

ine_match_alta_export (98, 11)
¿Tiene vinculo_grupo? False

bv_match_alta_export (3824, 11)
¿Tiene vinculo_grupo? False

ine_match_alta_media_export (128, 11)
¿Tiene vinculo_grupo? False

bv_match_alta_media_export (34782, 11)
¿Tiene vinculo_grupo? False



In [22]:
# Guardar bases de matching para revisión previa

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ine_match_alta_export.to_csv(
    OUTPUT_DIR / "ine_match_alta_confianza_2020_2022.csv",
    index=False,
    encoding="utf-8-sig"
)

bv_match_alta_export.to_csv(
    OUTPUT_DIR / "banavim_match_alta_confianza_2020_2022.csv.gz",
    index=False,
    encoding="utf-8-sig",
    compression="gzip"
)

ine_match_alta_media_export.to_csv(
    OUTPUT_DIR / "ine_match_alta_media_confianza_2020_2022.csv",
    index=False,
    encoding="utf-8-sig"
)

bv_match_alta_media_export.to_csv(
    OUTPUT_DIR / "banavim_match_alta_media_confianza_2020_2022.csv.gz",
    index=False,
    encoding="utf-8-sig",
    compression="gzip"
)

# Muestras ligeras para Data Wrangler
bv_match_alta_export.sample(
    n=min(50000, len(bv_match_alta_export)),
    random_state=42
).to_csv(
    OUTPUT_DIR / "banavim_match_alta_confianza_sample_50000.csv",
    index=False,
    encoding="utf-8-sig"
)

bv_match_alta_media_export.sample(
    n=min(50000, len(bv_match_alta_media_export)),
    random_state=42
).to_csv(
    OUTPUT_DIR / "banavim_match_alta_media_confianza_sample_50000.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_bases_match.to_csv(
    OUTPUT_DIR / "resumen_bases_match_confianza.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Bases guardadas en:", OUTPUT_DIR)

Bases guardadas en: ..\Data\fusion_outputs
